# Assignment: the two colors of galaxies

## The phenomenon

Galaxies come in every size and shape, but their colors are highly bimodal: most galaxies are either blue or red, with only a few in between.

Color is an indicator of the star formation activity in a galaxy. Blue light is primarily emitted by hot, massive stars with a life-span of only a few million years, so a galaxy stays blue only during its active star formation period. Once that star formation stops, the hot stars die out and the galaxy turns red. The visible gap between the blue and the red galaxies suggests that star formation does not fade slowly, but rather switches off quickly&mdash;compared with how long a galaxy spends on either side. What switches it off is still a matter of debate in astronomy. Hypotheses include phenomena produced by the large mass of the galaxy, its surroundings, collisions with other galaxies or its central black hole. Some of these suspects, collisions above all, also change the shape of a galaxy.

You will test both ideas with the nearby galaxies of the SDSS:

- **Part a.** Do the colors of galaxies split into two groups, and where is the gap?
- **Part b.** Does a galaxy's color go together with its shape?

## Before you start

- **What you practice:** counting on the server with `FLOOR` and `GROUP BY`, so that only a histogram crosses the network; conditional counting with `CASE`; joining spectra to imaging; missing values.
- **Background reading:** sections 1, 4, 5 and 8 of the intro notebook, `intro-casjobs-and-functions.ipynb`.
- **How to work:** each step below says what to do. Steps marked **Report** ask for answers in words and numbers.
- **The server:** the queries need no SciServer account. CasJobs is shared: a query that usually takes seconds can now and then hang, or stop with "Query exceeds queue time". Interrupt it and run it again.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from SciServer import CasJobs

# The queries in this notebook need no SciServer login. Silence the
# warning SciScript gives about the missing login token.
warnings.filterwarnings("ignore", message="In Authentication.getToken")

## Background

- **Color.** The color $u - r$ is the difference of a galaxy's magnitudes in the $u$ (near-ultraviolet) and $r$ (red) filters. Magnitudes grow as light gets fainter, so a larger $u - r$ means less ultraviolet light relative to red light: a redder galaxy.
- **Concentration.** $C = r_{90} / r_{50}$ is the ratio of the radii that hold $90\,\%$ and $50\,\%$ of a galaxy's light. Light that crowds toward the center, as in an elliptical galaxy, gives a large $C$; the more evenly spread light of a spiral disk gives a smaller one.
- **Reference values.** Strateva et al. (2001) split galaxies at $u - r = 2.22$ and at $C = 2.6$. The red, concentrated side holds mostly ellipticals, lenticulars and spirals with large central bulges; the blue, less concentrated side mostly spirals with smaller bulges and irregular galaxies.

## Data

**Queries.** `CasJobs.executeQuery(sql=query, context=CASJOBS_CONTEXT, format="pandas")` runs a query and returns its result as a pandas DataFrame (intro, section 1).

**Tables.** `SpecObj` (one row per spectrum) joined to `PhotoObj` (one row per object in the images) on `p.objID = s.bestObjID` (intro, section 4).

**Sample.** The Main Galaxy Sample of SDSS-I/II (intro, section 5):

- `(s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0`
- `s.class = 'GALAXY'`
- `s.zWarning = 0`
- `s.z` between `Z_MIN` and `Z_MAX`

Do not require `p.clean = 1`: its conditions were chosen for point sources, not galaxies (intro, section 5).

**Columns** of `PhotoObj`:

| Column | Meaning |
| --- | --- |
| `dered_u`, `dered_r` | model magnitudes, corrected for the dust of the Milky Way |
| `petroR50_r`, `petroR90_r` | radii, in arcseconds, that hold $50\,\%$ and $90\,\%$ of the galaxy's Petrosian flux in the $r$ band |

**Parameters**, set in the next cell:

| Name | Meaning |
| --- | --- |
| `CASJOBS_CONTEXT` | release of the SDSS database that the queries run in |
| `Z_MIN`, `Z_MAX` | redshift range |
| `COLOR_MIN`, `COLOR_MAX`, `COLOR_STEP` | range of $u - r$ and width of its bins |
| `CONCENTRATION_MIN`, `CONCENTRATION_MAX`, `CONCENTRATION_STEP` | range of $C$ and width of its bins |
| `COLOR_SPLIT`, `CONCENTRATION_SPLIT` | the reference values of Strateva et al. |

**Missing values.** A missing magnitude or radius is stored as $-9999$. The ranges of color and concentration remove such galaxies: a $-9999$ magnitude puts $u - r$ near $\pm 10000$, and two $-9999$ radii give $C = 1$.

**Colors are observed.** The expansion of the Universe shifts a galaxy's light to longer wavelengths, which changes its observed color a little. The colors here are not corrected for this shift; Strateva et al. used observed colors too, and at these low redshifts the shift is small.

In [ ]:
# Release of the SDSS database that the queries run in.
CASJOBS_CONTEXT = "DR17"

# Redshift range of the sample.
Z_MIN = 0.02
Z_MAX = 0.08

# Range of the color u - r and width of its bins.
COLOR_MIN = 0.5
COLOR_MAX = 4.0
COLOR_STEP = 0.05

# Range of the concentration C and width of its bins.
CONCENTRATION_MIN = 1.5
CONCENTRATION_MAX = 4.0
CONCENTRATION_STEP = 0.05

# Splits between early and late types (Strateva et al. 2001).
COLOR_SPLIT = 2.22
CONCENTRATION_SPLIT = 2.6

## Part a. The color distribution

Goal: a histogram of $u - r$ for the whole sample, counted on the server.

**a1. Write the sample in SQL.** Write the `FROM` and `WHERE` clauses as a Python string, with the join, the four sample conditions and `COLOR_MIN` $\leq u - r <$ `COLOR_MAX`. Part b reuses this string.

**a2. Number the color bins.** Write the SQL expression for each galaxy's bin number: `FLOOR((u - r - COLOR_MIN) / COLOR_STEP)`. Bin $0$ starts at `COLOR_MIN`.

**a3. Count the galaxies per bin on the server.** Combine a1 and a2 into one query that selects the bin number and `COUNT(*)`, grouped by the bin number (intro, section 8). SQL Server cannot `GROUP BY` an alias from the `SELECT` list, so repeat the expression. Run the query, then print the number of rows and the total number of galaxies.

**a4. Fill in the empty bins.** A bin without galaxies has no row in the result. Build the counts of all bins, with zero where no row came back (`reindex`), and the bin edges (`np.linspace`).

**a5. Draw the histogram.** Plot the galaxies per bin against $u - r$, with axis labels and a dashed vertical line at `COLOR_SPLIT`.

**a6. Describe your findings.**

1. How many galaxies did the server count, and how many rows did it send back?
2. At which $u - r$ do the two peaks lie?
3. Where is the dip between them, and how does it compare with $2.22$?
4. How deep is the dip compared with the blue peak? What does that mean for sorting galaxies by color alone?

## Part b. Color and shape

Goal: a 2D histogram of concentration against color, and the fraction of concentrated galaxies among the red and the blue ones.

**b1. Add the concentration to the sample.** Extend the string of a1 with `CONCENTRATION_MIN` $\leq C <$ `CONCENTRATION_MAX`, where $C$ is `p.petroR90_r / p.petroR50_r`.

**b2. Count the galaxies per cell on the server.** Number the concentration bins as in a2, `CONCENTRATION_STEP` wide from `CONCENTRATION_MIN`. Select both bin numbers and `COUNT(*)`, and group by both. Run the query and print the number of rows and of galaxies.

**b3. Fill a 2D array.** Make an array with one row per color bin and one column per concentration bin. Put each count in its cell, and leave zero in the cells without galaxies.

**b4. Draw the 2D histogram.** Plot concentration against $u - r$, shaded by the number of galaxies on a logarithmic scale (`pcolormesh` with `LogNorm`). Add a colorbar and dashed lines at `COLOR_SPLIT` and `CONCENTRATION_SPLIT`.

**b5. Count the color groups on the server.** Write one query on the sample of b1 that returns one row per color group, with three columns:

- `color_group`: `'red'` where $u - r \geq$ `COLOR_SPLIT`, otherwise `'blue'`;
- `galaxies`: the number of galaxies in the group;
- `concentrated`: the number of them with $C \geq$ `CONCENTRATION_SPLIT`.

A `CASE` expression turns a condition into a value: `CASE WHEN ... THEN 'red' ELSE 'blue' END` names the group, and `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` counts the rows that meet a condition.

**b6. Compute the fractions.** For each color group, divide the concentrated galaxies by all galaxies.

**b7. Describe your findings.**

1. Do the red and the blue galaxies form separate clumps in the 2D histogram? Around which $C$ does each clump lie?
2. What fraction of the red galaxies is concentrated, and what fraction of the blue ones?
3. Does color or concentration alone sort the galaxies cleanly? Look at the red galaxies below the concentration split and the blue ones above it.